In [0]:
storage_account_name = "stfrauddatabricks"
container_gold = "gold"
project_folder = "fraud_detection"

gold_base_path = f"abfss://{container_gold}@{storage_account_name}.dfs.core.windows.net/{project_folder}"

print("GOLD path:", gold_base_path)

In [0]:
# definir catálogo y schema


catalog_name = "hive_metastore"
schema_name = "fraud_gold"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema_name}")
spark.sql(f"USE {schema_name}")

print(f"Catálogo usado: {catalog_name}")
print(f"Base de datos / schema activo: {schema_name}")

In [0]:


gold_tables = {
    "gold_kpi_fraud_summary": f"{gold_base_path}/gold_kpi_fraud_summary_delta",
    "gold_fraud_by_transaction_type": f"{gold_base_path}/gold_fraud_by_transaction_type_delta",
    "gold_fraud_by_risk_level": f"{gold_base_path}/gold_fraud_by_risk_level_delta",
    "gold_fraud_by_amount_category": f"{gold_base_path}/gold_fraud_by_amount_category_delta",
    "gold_fraud_by_profile_risk": f"{gold_base_path}/gold_fraud_by_profile_risk_delta",
    "gold_top_suspicious_accounts": f"{gold_base_path}/gold_top_suspicious_accounts_delta",
    "gold_fraud_by_step": f"{gold_base_path}/gold_fraud_by_step_delta",
    "gold_dashboard_executive_summary": f"{gold_base_path}/gold_dashboard_executive_summary_delta"
}

for table_name, table_path in gold_tables.items():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {schema_name}.{table_name}
        USING DELTA
        LOCATION '{table_path}'
    """)
    print(f"Tabla registrada: {schema_name}.{table_name}")

In [0]:
# COMMAND ----------

spark.sql(f"SHOW TABLES IN {schema_name}").show(truncate=False)

In [0]:
display(spark.sql(f"""
SELECT *
FROM {schema_name}.gold_kpi_fraud_summary
"""))

In [0]:
display(spark.sql(f"""
SELECT 
    transaction_type,
    risk_level,
    total_transactions,
    fraud_transactions,
    fraud_amount,
    fraud_rate_pct
FROM {schema_name}.gold_fraud_by_transaction_type
ORDER BY fraud_transactions DESC
"""))

In [0]:
display(spark.sql(f"""
SELECT 
    origin_account_id,
    origin_customer_segment,
    origin_kyc_status,
    total_transactions,
    fraud_transactions,
    fraud_amount,
    suspicious_account_score
FROM {schema_name}.gold_top_suspicious_accounts
ORDER BY suspicious_account_score DESC
LIMIT 20
"""))